In [1]:
# NORTHSTAR — Solace Browser: Security Hackers QA (Tower 6)
# DNA: security_qa = xss(escape+csp) x cors(allowlist) x auth(oauth3+budget+scope) x crypto(aes+sha) x evidence(chain+seal)
# Towers: T1(Masters) T6(Hackers) T17(Performance) T23(Web)
# Auth: 65537 | File-based probes — reads source directly
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-security-hackers-qa"
NOTEBOOK_PATH = "notebooks/qa/01-security-hackers-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
SRC = BROWSER / "src"
JS_DIR = WEB / "js"
CSS_DIR = WEB / "css"

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_src(path):
    return path.read_text(encoding="utf-8") if path.exists() else ""

print(f"BOOTSTRAP: Security QA for solace-browser")
print(f"  web/: {sum(1 for _ in WEB.rglob('*.html'))} HTML, {sum(1 for _ in JS_DIR.rglob('*.js'))} JS")
print(f"  src/: {sum(1 for _ in SRC.rglob('*.py'))} Python")

BOOTSTRAP: Security QA for solace-browser
  web/: 23 HTML, 16 JS
  src/: 141 Python


In [2]:
# P1: XSS Defense
print("=== P1: XSS Defense ===")
html_files = list(WEB.glob("*.html"))
full_pages = [f for f in html_files if not f.name.startswith("partials")]
csp_count = sum(1 for f in full_pages if "Content-Security-Policy" in read_src(f))
csp_pct = csp_count / max(len(full_pages), 1) * 100
record("P1-csp-headers", f"CSP on HTML pages ({csp_count}/{len(full_pages)}, {csp_pct:.0f}%)", csp_pct >= 80, tower_ref="T6")

js_files = list(JS_DIR.glob("*.js"))
has_escape = any("escapeHtml" in read_src(f) for f in js_files)
record("P1-escape-html", "escapeHtml defense function exists", has_escape, tower_ref="T6")

eval_files = []
for js_file in js_files:
    if js_file.stat().st_size > 500000: continue
    js_src = read_src(js_file)
    if re.search(r'\beval\s*\(', js_src) or re.search(r'document\.write\s*\(', js_src):
        eval_files.append(js_file.name)
record("P1-no-eval", f"No eval/document.write ({len(eval_files)} files)", len(eval_files) == 0, tower_ref="T6")

for f in full_pages:
    html = read_src(f)
    m = re.search(r'Content-Security-Policy["\']\s+content=["\']([^"\']+)', html)
    if m:
        record("P1-no-unsafe-eval", "CSP no unsafe-eval", "unsafe-eval" not in m.group(1), tower_ref="T6")
        break
print(f"\nP1: {sum(1 for r in results if r['id'].startswith('P1') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P1'))}")

=== P1: XSS Defense ===
  PASS: CSP on HTML pages (17/18, 94%)
  PASS: escapeHtml defense function exists
  PASS: No eval/document.write (0 files)
  PASS: CSP no unsafe-eval

P1: 4/4


In [3]:
# P2: CORS + CSRF + Clickjacking
print("=== P2: CORS + CSRF ===")
# Server is at web/server.py
server_py = WEB / "server.py"
server_src = read_src(server_py)
if not server_src:
    alt = list(SRC.rglob("*server*.py")) + list(SRC.rglob("*http*.py"))
    server_src = read_src(alt[0]) if alt else ""

record("P2-no-wildcard-cors", "No wildcard CORS", not bool(re.search(r'Access-Control-Allow-Origin.*\*', server_src)), tower_ref="T6")
record("P2-origin-allowlist", "CORS origin allowlist", bool(re.search(r'localhost|127\.0\.0\.1|ALLOWED_ORIGIN', server_src, re.IGNORECASE)), tower_ref="T6")
record("P2-clickjacking", "Frame protection (CSP frame-ancestors)", any("frame-ancestors" in read_src(f) for f in full_pages[:5]), tower_ref="T6")

# Path traversal: check for path validation (translate_path, safe_path, or resolve)
has_path_defense = bool(re.search(r'translate_path|safe_path|os\.path\.abspath|\.resolve\(\)|realpath|normpath', server_src))
record("P2-path-traversal", "Path traversal defense", has_path_defense, tower_ref="T6")
print(f"\nP2: {sum(1 for r in results if r['id'].startswith('P2') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P2'))}")

=== P2: CORS + CSRF ===
  PASS: No wildcard CORS
  PASS: CORS origin allowlist
  PASS: Frame protection (CSP frame-ancestors)
  PASS: Path traversal defense

P2: 4/4


In [4]:
# P3: OAuth3 + Budget Gates
print("=== P3: OAuth3 + Auth ===")
oauth3_dir = SRC / "oauth3"
oauth3_files = list(oauth3_dir.rglob("*.py")) if oauth3_dir.exists() else []
record("P3-oauth3-module", f"OAuth3 module ({len(oauth3_files)} files)", len(oauth3_files) >= 1, tower_ref="T6")

oauth3_src = "\n".join(read_src(f) for f in oauth3_files)
record("P3-scope-gate", "Scope enforcement exists", "ScopeGate" in oauth3_src or "scope_check" in oauth3_src, tower_ref="T6")

budget_path = SRC / "budget_gates.py"
if budget_path.exists():
    bsrc = read_src(budget_path)
    record("P3-budget-gates", "Budget gates fail-closed", "check_all" in bsrc and bool(re.search(r'allowed.*False|block', bsrc, re.IGNORECASE)), tower_ref="T6")
else:
    record("P3-budget-gates", "Budget gates exist", False, tower_ref="T6")

token_files = list(SRC.rglob("*token*.py"))
record("P3-token-revocation", "Token revocation exists", any("revoke" in read_src(f).lower() for f in token_files), tower_ref="T6")

# Rate limiting: check web/server.py (already loaded as server_src)
has_rate = bool(re.search(r'rate.?limit|throttl|_rate_limit|per_minute', server_src, re.IGNORECASE))
record("P3-rate-limiting", "Rate limiting exists", has_rate, tower_ref="T6")
print(f"\nP3: {sum(1 for r in results if r['id'].startswith('P3') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P3'))}")

=== P3: OAuth3 + Auth ===
  PASS: OAuth3 module (9 files)
  PASS: Scope enforcement exists
  PASS: Budget gates fail-closed
  PASS: Token revocation exists
  PASS: Rate limiting exists

P3: 5/5


In [5]:
# P4: Crypto + Vault
print("=== P4: Crypto + Vault ===")
vault_files = list(SRC.rglob("*vault*.py")) + list(SRC.rglob("*crypto*.py")) + list(SRC.rglob("*encrypt*.py"))
vault_src = "\n".join(read_src(f) for f in vault_files)
record("P4-encryption", "AES/Fernet encryption", bool(re.search(r'AES|AESGCM|Fernet', vault_src, re.IGNORECASE)), tower_ref="T6")

all_py = "\n".join(read_src(f) for f in SRC.rglob("*.py") if f.stat().st_size < 200000)
record("P4-sha256", "SHA-256 hashing", "sha256" in all_py or "SHA256" in all_py, tower_ref="T6")
record("P4-hmac", "HMAC signing", "hmac" in all_py.lower(), tower_ref="T6")
record("P4-no-secrets", "No hardcoded secrets", len(re.findall(r'(?:password|secret|api_key)\s*=\s*["\'][A-Za-z0-9]{20,}', all_py)) == 0, tower_ref="T6")
record("P4-deps", "Dependency file exists", (BROWSER / "requirements.txt").exists() or (BROWSER / "pyproject.toml").exists(), tower_ref="T6")
print(f"\nP4: {sum(1 for r in results if r['id'].startswith('P4') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P4'))}")

=== P4: Crypto + Vault ===
  PASS: AES/Fernet encryption
  PASS: SHA-256 hashing
  PASS: HMAC signing
  PASS: No hardcoded secrets
  PASS: Dependency file exists

P4: 5/5


In [6]:
# P5: Evidence + Security Docs
print("=== P5: Evidence + Docs ===")
evidence_files = list(SRC.rglob("*evidence*.py")) + list(SRC.rglob("*chain*.py"))
evidence_src = "\n".join(read_src(f) for f in evidence_files)
record("P5-evidence-chain", "Evidence chain exists", bool(re.search(r'seal|hash_chain|sha256', evidence_src, re.IGNORECASE)), tower_ref="T6")

# Security headers: check web/server.py
record("P5-sec-headers", "Security headers", bool(re.search(r'X-Content-Type|Strict-Transport|X-Frame-Options|Referrer-Policy', server_src)), tower_ref="T6")

sec_doc = (BROWSER / "SECURITY.md").exists() or (WEB / "security.txt").exists()
record("P5-security-doc", "SECURITY.md or security.txt", sec_doc, tower_ref="T6")

sec_txt = WEB / "security.txt"
if sec_txt.exists():
    record("P5-vuln-disclosure", "security.txt has contact", bool(re.search(r'Contact|mailto|https', read_src(sec_txt))), tower_ref="T6")
else:
    record("P5-vuln-disclosure", "Disclosure policy", sec_doc, tower_ref="T6")

broad = sum(len(re.findall(r'except\s+Exception\s*:', read_src(f))) for f in SRC.rglob("*.py"))
record("P5-fallback-ban", f"Broad except in src/ ({broad})", broad <= 30, tower_ref="T6")
print(f"\nP5: {sum(1 for r in results if r['id'].startswith('P5') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P5'))}")

=== P5: Evidence + Docs ===
  PASS: Evidence chain exists
  PASS: Security headers
  PASS: SECURITY.md or security.txt
  PASS: Disclosure policy
  PASS: Broad except in src/ (4)

P5: 5/5


In [7]:
# EVIDENCE SUMMARY
print("=== Evidence Summary ===\n")
total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = total - passed
score = round((passed / total) * 100, 1) if total > 0 else 0.0
print(f"  Total probes : {total}")
print(f"  Passed       : {passed}")
print(f"  Failed       : {failed}")
print(f"  Score        : {score}%")
print(f"  MIN_SCORE    : {MIN_SCORE}%")
if failed > 0:
    print("\nFAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']}" + (f" -- {r['detail']}" if r['detail'] else ""))
evidence = json.dumps(results, sort_keys=True)
evidence_hash = hashlib.sha256(evidence.encode()).hexdigest()[:16]
print(f"\n  Evidence hash: {evidence_hash}")
print(f"  Timestamp:    {datetime.now().isoformat()}")
assert score >= MIN_SCORE, f"Score {score}% below minimum {MIN_SCORE}%"
print(f"\nVERDICT: PASS ({score}% >= {MIN_SCORE}% minimum)")

=== Evidence Summary ===

  Total probes : 23
  Passed       : 23
  Failed       : 0
  Score        : 100.0%
  MIN_SCORE    : 70%

  Evidence hash: 9f7500089804d333
  Timestamp:    2026-03-06T10:24:33.972389

VERDICT: PASS (100.0% >= 70% minimum)
